# Thresholded `p_correct` summaries

This notebook reports, for each dataset, the fraction of problems whose per-problem correctness probability exceeds fixed thresholds. The three columns compare the canonical sampling condition (`pass@k`), retokenization (`pass@retok`), and typo perturbations (`pass@typo`).

In [ ]:
from __future__ import annotations

import pandas as pd

from eval import load, passat


MODEL_NAME = "allenai/OLMo-2-1124-7B-Instruct"

DATASET_CONFIGS = {
    "humaneval": {"dataset_size": 164, "default_numvariants": 51},
    "gsm8k": {"dataset_size": 1000, "default_numvariants": 30},
    "gsm8k_python": {"dataset_size": 1000, "default_numvariants": 11},
    "mmlu": {"dataset_size": 1000, "default_numvariants": 30},
}

DATASETS = ["humaneval", "gsm8k", "gsm8k_python", "mmlu"]
VARIANTS = ["temperature", "retok", "typo"]

DATASET_LABELS = {
    "humaneval": "HumanEval",
    "gsm8k": "GSM8K",
    "gsm8k_python": "GSM8K Python",
    "mmlu": "MMLU",
}

VARIANT_LABELS = {
    "temperature": "pass@k fraction",
    "retok": "pass@retok fraction",
    "typo": "pass@typo fraction",
}

THRESHOLDS = [0.0, 0.5, 0.75]

LOADERS = {
    "humaneval": load.load_humaneval,
    "gsm8k": load.load_gsm8k,
    "gsm8k_python": load.load_gsm8k_python,
    "mmlu": load.load_mmlu,
}


def requested_numvariants(dataset: str, variant: str) -> int:
    if dataset == "gsm8k_python" or variant == "typo":
        return 11
    return int(DATASET_CONFIGS[dataset]["default_numvariants"])


def load_variant(dataset: str, variant: str) -> pd.DataFrame:
    config = DATASET_CONFIGS[dataset]
    return LOADERS[dataset](
        model_name=MODEL_NAME,
        dataset_size=int(config["dataset_size"]),
        numvariants=requested_numvariants(dataset, variant),
        variant_type=variant,
    )


In [ ]:
data = {
    dataset: {variant: load_variant(dataset, variant) for variant in VARIANTS}
    for dataset in DATASETS
}


In [ ]:
def task_p_correct(df: pd.DataFrame) -> pd.Series:
    """Return one p_correct value per task for a loaded result dataframe."""
    dataset = df.attrs.get("dataset")
    variant_type = df.attrs.get("variant_type")

    if dataset == "mmlu" and variant_type == "temperature":
        summary = passat.summarize_task_answer_probabilities(df)
        return summary.set_index("task_id")["answer_prob"].astype(float).sort_index()

    summary = passat.summarize_task_outcomes(df)
    p_correct = summary["num_correct"] / summary["num_samples"]
    return pd.Series(p_correct.to_numpy(dtype=float), index=summary["task_id"], name="p_correct").sort_index()


def threshold_fraction(p_correct: pd.Series, threshold: float) -> float:
    return float((p_correct > threshold).mean())


def threshold_summary_table(dataset: str) -> pd.DataFrame:
    p_correct_by_variant = {
        variant: task_p_correct(data[dataset][variant])
        for variant in VARIANTS
    }

    rows = []
    for threshold in THRESHOLDS:
        row = {"p_correct threshold": f"p_correct > {threshold:g}"}
        for variant in VARIANTS:
            row[VARIANT_LABELS[variant]] = threshold_fraction(p_correct_by_variant[variant], threshold)
        rows.append(row)

    return pd.DataFrame(rows).set_index("p_correct threshold")


summary_tables = {dataset: threshold_summary_table(dataset) for dataset in DATASETS}
summary_tables_percent = {
    dataset: table.style.format("{:.1%}")
    for dataset, table in summary_tables.items()
}


## HumanEval

In [ ]:
summary_tables_percent["humaneval"]


## GSM8K

In [ ]:
summary_tables_percent["gsm8k"]


## GSM8K Python

In [ ]:
summary_tables_percent["gsm8k_python"]


## MMLU

In [ ]:
summary_tables_percent["mmlu"]
